# Lab 18 solution, notebook 2: ten customers, and a new label on the form

The worked answers to the two challenge tasks of `18b_agent_designs_at_the_desk.ipynb`. The setup repeats the
notebook's lookups, agents and router, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import asyncio
import os
import sqlite3
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import AsyncOpenAI
from openai.types.shared import Reasoning
from agents import Agent, ModelSettings, Runner, function_tool, set_default_openai_client

set_default_openai_client(AsyncOpenAI(timeout=60, max_retries=2))
SETTINGS = ModelSettings(reasoning=Reasoning(effort=EFFORT))

In [3]:
from contextlib import closing

DB_URI = f"file:{ROOT / 'data' / 'chinook.db'}?mode=ro"


def query(sql, args):
    """Run one read-only query on a connection of its own, and return every row."""
    with closing(sqlite3.connect(DB_URI, uri=True)) as conn:
        return conn.execute(sql, args).fetchall()


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = query("SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
                 "WHERE FirstName || ' ' || LastName = ?", (name,))
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = query("SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,))[0]
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    rows = query("SELECT e.FirstName || ' ' || e.LastName FROM customers c "
                 "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,))
    return {"customer_id": customer_id, "support_rep": rows[0][0] if rows else None}

In [4]:
desk = Agent(name="Desk", model=MODEL, model_settings=SETTINGS,
             tools=[function_tool(find_customer), function_tool(get_invoices), function_tool(get_support_rep)],
             instructions=("You help the support reps of a music shop. Look up every fact with the tools; "
                           "never answer from memory. Answer in one or two sentences."))
worker = Agent(name="Customer worker", model=MODEL, model_settings=SETTINGS,
               tools=[function_tool(find_customer), function_tool(get_invoices)],
               instructions=("You look up one customer's total spend. Look up every fact with your tools; never "
                             "answer from memory. Reply with one line: the name and the total in dollars."))
main = Agent(name="Main", model=MODEL, model_settings=SETTINGS,
             instructions="You help the support reps of a music shop. Answer from the lines you are given; do not add facts.")
print([agent.name for agent in [desk, worker, main]])

['Desk', 'Customer worker', 'Main']


## Challenge 1: grow the request to ten customers

The five customers of the notebook and five more. The same two designs; for each we print the answer and its
answering request.

In [5]:
TEN = ["Helena Holý", "Puja Srivastava", "Richard Cunningham", "Luis Rojas", "Manoj Pareek",
       "Ladislav Kovács", "Hugh O'Reilly", "Frank Ralston", "Astrid Gruber", "Terhi Hämäläinen"]
REQUEST = ("What has each of these ten customers spent in total: " + ", ".join(TEN[:-1]) + " and " + TEN[-1]
           + "? Who spent the most?")

for name in TEN:
    print(f"{name:20} ${get_invoices(find_customer(name)[0]['customer_id'])['total']:.2f}")

Helena Holý          $49.62
Puja Srivastava      $36.64
Richard Cunningham   $47.62
Luis Rojas           $46.62
Manoj Pareek         $38.62
Ladislav Kovács      $45.62
Hugh O'Reilly        $45.62
Frank Ralston        $43.62
Astrid Gruber        $42.62
Terhi Hämäläinen     $41.62


In [6]:
alone = await Runner.run(desk, REQUEST)
print(alone.final_output)
print("one agent: requests", alone.context_wrapper.usage.requests,
      "| answering request", alone.raw_responses[-1].usage.input_tokens)

Helena Holý: $49.62; Puja Srivastava: $36.64; Richard Cunningham: $47.62; Luis Rojas: $46.62; Manoj Pareek: $38.62; Ladislav Kovács: $45.62; Hugh O'Reilly: $45.62; Frank Ralston: $43.62; Astrid Gruber: $42.62; Terhi Hämäläinen: $41.62. **Helena Holý spent the most**, at $49.62.
one agent: requests 3 | answering request 1287


In [7]:
worker_runs = await asyncio.gather(*[Runner.run(worker, f"What has {name} spent in total?") for name in TEN])
lines = "\n".join(run.final_output for run in worker_runs)
joined = await Runner.run(main, REQUEST + "\n\nWhat the workers found:\n" + lines)
print(joined.final_output)
print("ten workers + main: requests", sum(run.context_wrapper.usage.requests for run in [*worker_runs, joined]),
      "| answering request", joined.raw_responses[-1].usage.input_tokens)

- Helena Holý — $49.62
- Puja Srivastava — $36.64
- Richard Cunningham — $47.62
- Luis Rojas — $46.62
- Manoj Pareek — $38.62
- Ladislav Kovács — $45.62
- Hugh O'Reilly — $45.62
- Frank Ralston — $43.62
- Astrid Gruber — $42.62
- Terhi Hämäläinen — $41.62

**Helena Holý spent the most: $49.62.**
ten workers + main: requests 31 | answering request 208


Both designs gave the ten right totals and named Helena Holý. Against the notebook's five customers:

| | Five customers | Ten customers |
|---|---|---|
| One agent, answering request | 756 tokens | 1,287 tokens |
| Main agent, answering request | 125 tokens | 208 tokens |
| One agent, requests | 3 | 3 |
| Workers and main, requests | 16 | 31 |

One agent's answering request grew by about 106 tokens per customer added (two calls and two results each); the main agent's by about 17 (one line). One agent still took 3 requests, because it asked for all ten lookups of each kind in one reply. The gap in what the answering agent reads grows with the work, and so does the split's cost in requests.

## Challenge 2: give the form a new label

The router of Part D with one of its handlers, and the new `country` handler. `find_customer` already returns the country, so the new
handler needs no new lookup.

In [8]:
def customer_id_of(name):
    return find_customer(name)[0]["customer_id"]


def answer_support_rep(request):
    rep = get_support_rep(customer_id_of(request["customer"]))["support_rep"]
    return f"{request['customer']}'s support rep is {rep}."


def answer_country(request):
    country = find_customer(request["customer"])[0]["country"]
    return f"{request['customer']} lives in {country}."


HANDLERS = {"support_rep": answer_support_rep, "country": answer_country}


async def route(request):
    handler = HANDLERS.get(request["kind"])
    if handler:                                    # a labelled request: our code answers
        return handler(request), 0
    run = await Runner.run(desk, request["text"])  # anything else goes to the agent
    return run.final_output, run.context_wrapper.usage.requests


for request in [{"kind": "country", "customer": "Puja Srivastava"},
                {"kind": "refund", "text": "Can Helena Holý get a refund for her last invoice?"}]:
    reply, requests = await route(request)
    print(f"[{request['kind']}] model requests: {requests}\n{reply}\n")

[country] model requests: 0
Puja Srivastava lives in India.



[refund] model requests: 3
I can’t determine refund eligibility from the available records; they show Helena has 7 invoices totaling $49.62, but no refund policy or invoice-specific status. Her support rep is Steve Johnson, who can check the last invoice.



The `country` request was answered by our code, *India*, with 0 model requests. `refund` is not a key of `HANDLERS`, so `route` sent the request to the desk agent. The agent has no refund tool: it looked Helena up (3 requests) and said it cannot settle a refund from the records it can read, and named her rep. A label with no handler goes to the agent: it costs an agent run, and it never reaches the wrong handler.